<a href="https://colab.research.google.com/github/Danzigerrr/MultiClass-Entity-Linking-System/blob/NER-datasets/NER_BERT_with_Conll2003.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BERT with Conll2003

Original code: https://github.com/rohan-paul/LLM-FineTuning-Large-Language-Models/blob/main/Other-Language_Models_BERT_related/YT_Fine_tuning_BERT_NER_v1.ipynb



**There are 9 types of labels in the dataset:**
- O means the word doesn’t correspond to any entity.
- B-PER/I-PER means the word corresponds to the beginning of/is inside a person entity.
- B-ORG/I-ORG means the word corresponds to the beginning of/is inside an organization entity.
- B-LOC/I-LOC means the word corresponds to the beginning of/is inside a location entity.
- B-MISC/I-MISC means the word corresponds to the beginning of/is inside a miscellaneous entity.



## Import libraries

In [1]:
!pip install datasets transformers tokenizers seqeval evaluate -q

In [2]:
import datasets
from datasets import load_dataset
import numpy as np
from transformers import BertTokenizerFast
from transformers import DataCollatorForTokenClassification
from transformers import AutoModelForTokenClassification

## Import Conll2003 dataset

In [3]:
# The dataset is stored at https://huggingface.co/datasets/eriktks/conll2003
conll2003 = load_dataset("conll2003", trust_remote_code=True)

In [4]:
conll2003

In [5]:
conll2003.shape

In [6]:
# first sample from train dataset:
conll2003["train"][0]

In [7]:
# feature names
conll2003["train"].features["ner_tags"]

## Create tokenizer

In [8]:
# define tokenizer
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

###  Problem of Sub-Token
The input ids returned by the tokenizer are longer than the lists of labels our dataset contain.
Therefore, we need to do some pre-processing on the data before training.
We need to depend on the result of the *word_ids()* mehtod.


In [9]:
example_text = conll2003['train'][0]

tokenized_input = tokenizer(example_text["tokens"], is_split_into_words=True)

tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])

word_ids = tokenized_input.word_ids()

print(f"word_ids: {word_ids}")

''' As we can see, it returns a list with the same number of
elements as our processed input ids, mapping special tokens to
None and all other tokens to their respective word.
This way, we can align the labels with the processed input ids. '''

print(f"tokenized_input: {tokenized_input}")

Length of ner_tags and input_ids are different:

In [10]:
len(example_text['ner_tags']), len(tokenized_input["input_ids"])
# (9, 11)

The below function tokenize_and_align_labels does 2 jobs

- set –100 as the label for these special tokens and the subwords we wish to mask during training
- mask the subword representations after the first subword

In [11]:
def tokenize_and_align_labels(examples, label_all_tokens=True):
    """
    Function to tokenize and align labels with respect to the tokens. This function is specifically designed for
    Named Entity Recognition (NER) tasks where alignment of the labels is necessary after tokenization.

    Parameters:
    examples (dict): A dictionary containing the tokens and the corresponding NER tags.
                     - "tokens": list of words in a sentence.
                     - "ner_tags": list of corresponding entity tags for each word.

    label_all_tokens (bool): A flag to indicate whether all tokens should have labels.
                             If False, only the first token of a word will have a label,
                             the other tokens (subwords) corresponding to the same word will be assigned -100.

    Returns:
    tokenized_inputs (dict): A dictionary containing the tokenized inputs and the corresponding labels aligned with the tokens.
    """
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        # word_ids() => Return a list mapping the tokens
        # to their actual word in the initial sentence.
        # It Returns a list indicating the word corresponding to each token.
        previous_word_idx = None
        label_ids = []
        # Special tokens like `<s>` and `<\s>` are originally mapped to None
        # We need to set the label to -100 so they are automatically ignored in the loss function.
        for word_idx in word_ids:
            if word_idx is None:
                # set –100 as the label for these special tokens
                label_ids.append(-100)
            # For the other tokens in a word, we set the label to either the current label or -100, depending on
            # the label_all_tokens flag.
            elif word_idx != previous_word_idx:
                # if current word_idx is != prev then its the most regular case
                # and add the corresponding token
                label_ids.append(label[word_idx])
            else:
                # to take care of sub-words which have the same word_idx
                # set -100 as well for them, but only if label_all_tokens == False
                label_ids.append(label[word_idx] if label_all_tokens else -100)
                # mask the subword representations after the first subword

            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels  # a new key is added
    return tokenized_inputs

In [12]:
q = tokenize_and_align_labels(conll2003['train'][0:1])
print(q)

In [13]:
for token, label in zip(tokenizer.convert_ids_to_tokens(q["input_ids"][0]),q["labels"][0]):
    print(f"{token:_<20} {label}")

In [14]:
# apply this method to the whole dataset
tokenized_datasets = conll2003.map(tokenize_and_align_labels, batched=True)

## Create the model

In [15]:
model = AutoModelForTokenClassification.from_pretrained("bert-base-uncased", num_labels=9)

In [16]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
  "test-ner",
  evaluation_strategy = "epoch",
  learning_rate=2e-5,
  per_device_train_batch_size=16,
  per_device_eval_batch_size=16,
  num_train_epochs=3,
  weight_decay=0.01,
)

In [17]:
# data_collators form the batch
data_collator = DataCollatorForTokenClassification(tokenizer)

### Metrics for evaluation

In [18]:
# import metrics
import datasets
import evaluate
metric = evaluate.load("seqeval")

In [19]:
sample_from_dataset = conll2003['train'][0]
sample_from_dataset

In [20]:
# 9 possible feature names (labels)
label_list = conll2003["train"].features["ner_tags"].feature.names

label_list

In [21]:
# check if the metric method is working:
labels = [label_list[i] for i in sample_from_dataset["ner_tags"]]

metric.compute(predictions=[labels], references=[labels])

In [22]:
def compute_metrics(eval_preds):
    """
    Function to compute the evaluation metrics for Named Entity Recognition (NER) tasks.
    The function computes precision, recall, F1 score and accuracy.

    Parameters:
    eval_preds (tuple): A tuple containing the predicted logits and the true labels.

    Returns:
    A dictionary containing the precision, recall, F1 score and accuracy.
    """
    pred_logits, labels = eval_preds

    pred_logits = np.argmax(pred_logits, axis=2)
    # the logits and the probabilities are in the same order,
    # so we don’t need to apply the softmax

    # We remove all the values where the label is -100
    predictions = [
        [label_list[eval_preds] for (eval_preds, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(pred_logits, labels)
    ]

    true_labels = [
      [label_list[l] for (eval_preds, l) in zip(prediction, label) if l != -100]
       for prediction, label in zip(pred_logits, labels)
   ]
    results = metric.compute(predictions=predictions, references=true_labels)
    return {
   "precision": results["overall_precision"],
   "recall": results["overall_recall"],
   "f1": results["overall_f1"],
  "accuracy": results["overall_accuracy"],
  }

## Train model

In [23]:
trainer = Trainer(
   model,
   args,
   train_dataset=tokenized_datasets["train"],
   eval_dataset=tokenized_datasets["validation"],
   data_collator=data_collator,
   tokenizer=tokenizer,
   compute_metrics=compute_metrics
)

In [24]:
trainer.train()

In [24]:
model.save_pretrained("ner_model")
tokenizer.save_pretrained("tokenizer")

In [25]:
id2label = {
    str(i): label for i,label in enumerate(label_list)
}
label2id = {
    label: str(i) for i,label in enumerate(label_list)
}

## Load trained model

In [26]:
import json

In [63]:
checkpoint_path = "/content/test-ner/checkpoint-2634"

In [30]:
config = json.load(open(checkpoint_path + "/config.json"))

In [31]:
config["id2label"] = id2label
config["label2id"] = label2id

In [33]:
json.dump(config, open(checkpoint_path + "/config.json","w"))

In [44]:
model_fine_tuned = AutoModelForTokenClassification.from_pretrained(checkpoint_path)
model_fine_tuned

In [37]:
from transformers import pipeline

In [60]:
nlp = pipeline("ner", model=model_fine_tuned, tokenizer=tokenizer)

example = "Michael Jordan is a player who plays for the Chicago Bulls."

ner_results = nlp(example)

print(ner_results)

Visualize the output:

In [61]:
from IPython.core.display import display, HTML

def visualize_ner_results_merged(text, ner_results):
    # Mapping entity types to colors for visualization
    entity_colors = {
        "PER": "darkblue",
        "ORG": "darkgreen",
        "LOC": "darkcoral",
        "MISC": "darkgoldenrodyellow"
    }

    # Merge contiguous entities
    merged_entities = []
    current_entity = None

    for entity in ner_results:
        entity_type = entity['entity'].split('-')[-1]  # Extract the main type (e.g., PER)

        if entity['entity'].startswith("B-"):
            # Start of a new entity
            if current_entity:  # Append the previous entity
                merged_entities.append(current_entity)
            current_entity = {
                "type": entity_type,
                "start": entity['start'],
                "end": entity['end']
            }
        elif entity['entity'].startswith("I-") and current_entity and current_entity['type'] == entity_type:
            # Continuation of the current entity
            current_entity["end"] = entity['end']
        else:
            # Append the current entity if it exists
            if current_entity:
                merged_entities.append(current_entity)
            current_entity = None  # Reset for next

    # Append the last entity
    if current_entity:
        merged_entities.append(current_entity)

    # Build the visualization HTML
    highlighted_text = ""
    last_end = 0
    for entity in merged_entities:
        entity_type = entity['type']
        color = entity_colors.get(entity_type, "lightgray")  # Default color if not mapped
        start, end = entity['start'], entity['end']

        # Append text before the entity
        highlighted_text += text[last_end:start]

        # Append the highlighted entity
        highlighted_text += f"<span style='background-color: {color}; padding: 2px; border-radius: 3px;'>{text[start:end]} ({entity_type})</span>"

        # Update the last end position
        last_end = end

    # Append remaining text
    highlighted_text += text[last_end:]

    # Display the result
    display(HTML(f"<div style='font-family: Arial, sans-serif; line-height: 1.6;'>{highlighted_text}</div>"))

# Input text and results
visualize_ner_results_merged(example, ner_results)


### See logits (probabilities)

In [62]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Load tokenizer and fine-tuned model
checkpoint_path = "/content/test-ner/checkpoint-2634"
model_fine_tuned = AutoModelForTokenClassification.from_pretrained(checkpoint_path)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

# Tokenize input
inputs = tokenizer(example, return_tensors="pt", truncation=True, is_split_into_words=False)
with torch.no_grad():
    outputs = model_fine_tuned(**inputs)

# Extract logits
logits = outputs.logits
# Apply softmax to get probabilities
probabilities = torch.nn.functional.softmax(logits, dim=-1)

# Get the tokenized input IDs
input_ids = inputs["input_ids"].squeeze()
tokens = tokenizer.convert_ids_to_tokens(input_ids)

# Get the labels (from the model's config)
label_map = model_fine_tuned.config.id2label

# Process each token's probabilities
results = []
for idx, token_probs in enumerate(probabilities.squeeze()):
    token = tokens[idx]
    token_probs_np = token_probs.cpu().numpy()
    token_probs_dict = {label_map[i]: token_probs_np[i] for i in range(len(token_probs_np))}

    results.append({
        "token": token,
        "probabilities": token_probs_dict
    })

# Print results
for result in results:
    print(f"\nToken: {result['token']}")
    for label, prob in result["probabilities"].items():
        print(f"  {label:_<8}: {prob:.4f}")


### Evaluate on test datastet

In [59]:
from transformers import Trainer, TrainingArguments, AutoModelForTokenClassification
from sklearn.metrics import classification_report
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
import torch

# Load dataset
conll2003 = load_dataset("conll2003", trust_remote_code=True)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

# Tokenize the test dataset
def tokenize_function(examples):
    return tokenizer(examples['tokens'], truncation=True, padding='max_length', is_split_into_words=True)

# Apply tokenization to the test split
tokenized_test = conll2003['test'].map(tokenize_function, batched=True)

# Format the tokenized dataset to have input IDs and labels
def format_examples(examples):
    labels = examples['ner_tags']
    # Padding label to match the max_length if necessary
    labels = [label + [0] * (tokenizer.model_max_length - len(label)) for label in labels]
    return {"labels": labels}

tokenized_test = tokenized_test.map(format_examples, batched=True)

# Load the fine-tuned model
model_fine_tuned = AutoModelForTokenClassification.from_pretrained(checkpoint_path)

# Define the metric function to evaluate performance
def compute_metrics(p):
    predictions, labels = p
    # Convert logits to predicted labels
    predictions = np.argmax(predictions, axis=-1)

    # Flatten arrays
    true_labels = labels.flatten()
    pred_labels = predictions.flatten()

    # Exclude padding labels (label = 0) for metrics calculation
    mask = true_labels != 0
    true_labels = true_labels[mask]
    pred_labels = pred_labels[mask]

    # Return classification report metrics
    return classification_report(true_labels, pred_labels, output_dict=True)

# Define training arguments for evaluation
evaluation_args = TrainingArguments(
    per_device_eval_batch_size=8,
    output_dir="./results",
    do_train=False,
    do_eval=True,
    evaluation_strategy="epoch",
    logging_dir="./logs",
)

# Initialize the Trainer
trainer = Trainer(
    model=model_fine_tuned,
    args=evaluation_args,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# Evaluate the model
eval_results = trainer.evaluate()

# Print evaluation results
print("Evaluation Results:", eval_results)
